# 01 — Audio Basics

WAV okuma, waveform çizimi, STFT tabanlı güç spectrogramı ve mel-spectrogram hesaplama.

## Ses verisi ve örnekleme
| Terim | Anlamı |
|---|---|
| WAV | Ses verisini ve örnekleme bilgilerini taşıyan dosya biçimi. WAV bir grafik değildir; her WAV aynı kodlamayı kullanmaz. Örnek dosya PCM kodlaması kullanır. |
| Sample / örnek | Dijital sesin bir zaman noktasındaki genlik değeri. |
| Sampling rate / örnekleme hızı (`sr`) | Saniyedeki örnek sayısı; birimi Hz. 24 000 Hz, saniyede 24 000 örnek demektir. |
| Amplitude / genlik | Sinyalin anlık değeri. Float PCM analizinde genellikle yaklaşık −1 ile +1 aralığında çalışılır. Fiziksel ses basıncıyla aynı değildir. |
| Frequency / frekans | Saniyedeki titreşim sayısı, Hz. Algılanan pitch ile ilişkili ama aynı kavram değil. |
| Mono / stereo | Bir / iki ses kanalı. Kanal ortalaması stereo bilgiyi kaybettirir; karşı fazdaki kanallar birbirini söndürebilir. |
| Bit depth | PCM örneğinin hassasiyeti: örneğin 16 bit. Örnekleme hızıyla farklıdır. |
| Duration / süre | `örnek sayısı / sr`, saniye. |
| Nyquist frekansı | `sr / 2`. Örnekleme öncesi bu sınırın üzerindeki içerik filtrelenmelidir; aksi halde aliasing oluşabilir. |
| Clipping | Genliğin temsil sınırında kesilmesi ve bozulması. Yalnızca tepe değeri ölçümü clipping'in kesin kanıtı değildir. |

Dosyanın `sr` değerini değiştirmek, sesi yeniden örneklemek değildir. Gerçek resampling yeni örnekler ve gerekli filtreleme ile yapılır. Dosyanın örnekleme hızı korunur.

In [ ]:
from pathlib import Path
import sys
import json
import csv
import platform
import hashlib
import importlib.metadata
from datetime import datetime, timezone
import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.display
import soundfile as sf
from IPython.display import Audio, display

print("Python:", sys.version.split()[0])
print("librosa:", librosa.__version__)
plt.rcParams.update({"figure.figsize": (11, 3), "axes.grid": False})
# Repo kökünden veya notebooks/ klasöründen çalıştırılabilir.
CWD = Path.cwd()
ROOT = CWD.parent if CWD.name == "notebooks" else CWD
AUDIO_PATH = None  # Örnek: ROOT / "data" / "raw" / "impact.wav"

# Her yeniden çalıştırma aynı deney dosyalarını günceller.
FIGURES = ROOT / "results" / "figures" / "01_audio_basics"
TABLES = ROOT / "results" / "tables" / "01_audio_basics"
EXPERIMENT = ROOT / "experiments" / "01_audio_basics"
for folder in [FIGURES, TABLES, EXPERIMENT]:
    folder.mkdir(parents=True, exist_ok=True)


## WAV dosyası

`AUDIO_PATH` bir dosya yolu olduğunda mevcut WAV okunur. `None` olduğunda 24 kHz örnekleme hızında üç saniyelik sentetik sinyal oluşturulur. Darbe başlangıçları 0.4, 1.2 ve 2.0 saniyedir. Her darbe 220 Hz ve 1400 Hz sinüs bileşenleri, gürültü ve üstel sönme zarfı içerir.

**Onset**, olayın başlangıç anıdır. **Decay**, başlangıçtan sonra genliğin azalmasıdır. `np.random.default_rng(42)` gürültü üretimini tekrarlanabilir kılar. Sinyal tepe genliği 0.8 olacak şekilde ölçeklenir ve PCM_16 WAV olarak yazılır.

In [ ]:
if AUDIO_PATH is None:
    demo_sr = 24000
    t = np.arange(3 * demo_sr) / demo_sr
    demo = np.zeros_like(t)
    rng = np.random.default_rng(42)
    for onset in [0.4, 1.2, 2.0]:
        tau = t - onset
        active = tau >= 0
        u = tau[active]
        demo[active] += np.exp(-u / 0.12) * (
            0.6 * np.sin(2 * np.pi * 220 * u)
            + 0.25 * np.sin(2 * np.pi * 1400 * u)
            + 0.15 * rng.standard_normal(len(u)))
    demo *= 0.8 / max(np.max(np.abs(demo)), 1e-12)
    AUDIO_PATH = ROOT / "data" / "generated" / "audio_basics_demo.wav"
    AUDIO_PATH.parent.mkdir(parents=True, exist_ok=True)
    sf.write(AUDIO_PATH, demo, demo_sr, subtype="PCM_16")
else:
    AUDIO_PATH = Path(AUDIO_PATH).expanduser()
    if not AUDIO_PATH.is_absolute():
        AUDIO_PATH = ROOT / AUDIO_PATH
if not AUDIO_PATH.is_file():
    raise FileNotFoundError(f"WAV bulunamadı: {AUDIO_PATH}")
print("Dosya:", AUDIO_PATH)


## WAV okuma ve oynatma

`sf.read` ses örneklerini `float32` dizi olarak okur. `always_2d=True`, dizi boyutunu `(örnek sayısı, kanal sayısı)` olarak tutar. `audio.mean(axis=1)` kanal ortalamasını alarak mono sinyal oluşturur.

`duration = len(y) / sr` kayıt süresini verir. `normalize=False`, oynatma sırasında otomatik tepe normalizasyonunu kapatır. Duyulan seviye cihazın ses ayarına da bağlıdır.

In [ ]:
info = sf.info(AUDIO_PATH)
audio, sr = sf.read(AUDIO_PATH, dtype="float32", always_2d=True)
if len(audio) == 0 or not np.isfinite(audio).all():
    raise ValueError("Dosya boş veya geçersiz örnek içeriyor.")
y = audio.mean(axis=1)
duration = len(y) / sr
peak = float(np.max(np.abs(y)))
rms = float(np.sqrt(np.mean(y.astype(np.float64)**2)))
print(info)
print(f"Kanal: {audio.shape[1]} | Süre: {duration:.3f} s | Nyquist: {sr/2:.0f} Hz")
print(f"Mono peak: {peak:.4f} | RMS: {rms:.4f}")
if peak == 0:
    print("Bu kayıt mono analizde sessiz; spektrumda anlamlı ses içeriği bekleme.")
if peak > 1:
    print("Float örnekler ±1 sınırını aşıyor; oynatmadan önce ölçeği kontrol et.")
else:
    display(Audio(y, rate=sr, normalize=False))


## Waveform

Waveform, sinyal genliğinin zamana göre değişimidir. Yatay eksen saniye, dikey eksen genliktir. Darbe başlangıçları ve sönme süreleri bu grafikte izlenir; frekans bileşenleri ayrı olarak gösterilmez.

**Peak**, en büyük mutlak genliktir. **RMS** (root mean square), ortalama karesel genliğin kareköküdür:

$\mathrm{RMS}=\sqrt{\frac{1}{N}\sum_n y[n]^2}$

RMS bir sinyal seviyesi göstergesidir; algısal loudness veya kalibre edilmiş dB SPL ölçümü değildir.

In [ ]:
time = np.arange(len(y)) / sr
fig, ax = plt.subplots()
ax.plot(time, y, linewidth=0.6)
ax.set(xlabel="Zaman (s)", ylabel="Genlik", title="Waveform")
ax.set_xlim(0, duration)
plt.tight_layout()
fig.savefig(FIGURES / "waveform.svg", bbox_inches="tight")
plt.show()


## Güç spectrogramı
**FFT**, bir sinyal parçasını frekans bileşenlerine ayırır. **STFT** (Short-Time Fourier Transform), bunu ses boyunca kaydırılan kısa pencerelerde yapar. **Spectrogram**, STFT büyüklüğünü veya karesini renklerle gösterir. Güç temsili `abs(STFT)**2` ile hesaplanır; fiziksel watt veya kalibre edilmiş güç yoğunluğu değildir.

| Ayar | İşlevi |
|---|---|
| `n_fft` | FFT uzunluğu. Pencere uzunluğu burada FFT uzunluğuna eşittir. |
| `window="hann"` | Pencere kenarlarını yumuşatıp spectral leakage'i azaltır. Leakage, sonlu bir parçanın frekans enerjisinin komşu kutulara yayılmasıdır. |
| `hop_length` | Ardışık pencereler arasındaki örnek sayısı. Küçük hop daha sık zaman noktaları verir. |
| Frame | Analiz edilen kısa ses parçası. |
| Bin | FFT'nin bir frekans kutusu. Kutular arası aralık `sr / n_fft`. |
| `center=True` | Kenarlara sıfırlar ekleyerek frame'leri zaman noktalarının üzerine merkezler. |

Uzun pencere yakın frekansları ayırmaya yardım eder ama hızlı olayları zamanda yayar. Kısa pencere bunun tersidir. Bin aralığı, her durumda gerçek ayırt etme çözünürlüğü değildir; pencere tipi ve uzunluğu da önemlidir.

**dB (decibel)** logaritmik bir oran: güç için $10\log_{10}(P/P_{ref})$. `ref=np.max` ile en güçlü hücre 0 dB olur, diğerleri negatiftir. Bu dB SPL değildir. Her grafiğin kendi maksimumuna göre renklenmesi farklı kayıtların mutlak seviyelerini karşılaştırmaya uygun değildir. Sessiz sinyalde 0 dB görünümü ses olduğu anlamına gelmez.

In [ ]:
N_FFT = 1024
HOP_LENGTH = 256
N_MELS = 64
print(f"Pencere: {1000*N_FFT/sr:.1f} ms | Hop: {1000*HOP_LENGTH/sr:.1f} ms")
print(f"FFT bin aralığı: {sr/N_FFT:.2f} Hz")
D = librosa.stft(y=y, n_fft=N_FFT, hop_length=HOP_LENGTH,
                 win_length=N_FFT, window="hann", center=True)
power = np.abs(D)**2
power_db = librosa.power_to_db(power, ref=np.max, top_db=80)
fig, ax = plt.subplots()
img = librosa.display.specshow(power_db, sr=sr, hop_length=HOP_LENGTH,
                               x_axis="time", y_axis="linear", ax=ax,
                               vmin=-80, vmax=0)
fig.colorbar(img, ax=ax, label="dB (kendi maksimumuna göre)")
ax.set(title="Power spectrogram", xlabel="Zaman (s)", ylabel="Frekans (Hz)")
plt.tight_layout()
fig.savefig(FIGURES / "spectrogram.svg", bbox_inches="tight")
plt.show()
print("STFT (frekans bin, frame):", D.shape)


## Mel-spectrogram

**Mel ölçeği**, frekans algısını yaklaşık temsil eden doğrusal olmayan bir ölçektir. Düşük frekanslarda daha sık, yüksek frekanslarda daha geniş aralıklı bantlar kullanılır.

**Mel filter bank**, FFT güç kutularını ağırlıklandırıp mel bantlarında birleştirir. `S=power`, önceden hesaplanan güç spectrogramını kullanır. `n_mels=64`, 64 mel bandı oluşturur; `fmax=sr/2` üst frekans sınırıdır. `power_to_db` dönüşümünden sonra log-mel temsili elde edilir.

Mel bantlarında birleştirme frekans ayrıntısını azaltır. Mel-spectrogram faz içermez; özgün waveform'un doğrudan karşılığı değildir.

In [ ]:
mel_power = librosa.feature.melspectrogram(
    S=power, sr=sr, n_fft=N_FFT, n_mels=N_MELS, fmin=0, fmax=sr/2)
mel_db = librosa.power_to_db(mel_power, ref=np.max, top_db=80)
fig, ax = plt.subplots()
img = librosa.display.specshow(mel_db, sr=sr, hop_length=HOP_LENGTH,
                               x_axis="time", y_axis="mel", fmin=0, fmax=sr/2,
                               ax=ax, vmin=-80, vmax=0)
fig.colorbar(img, ax=ax, label="dB (kendi maksimumuna göre)")
ax.set(title="Log-mel spectrogram", xlabel="Zaman (s)", ylabel="Frekans (Hz, mel ölçeği)")
plt.tight_layout()
fig.savefig(FIGURES / "mel_spectrogram.svg", bbox_inches="tight")
plt.show()
print("Mel (mel bandı, frame):", mel_power.shape)
assert power.shape[0] == N_FFT // 2 + 1
assert mel_power.shape == (N_MELS, power.shape[1])
assert np.isfinite(power_db).all() and np.isfinite(mel_db).all()


## Genlik ölçekleme ve RMS

`quieter = 0.5 * y` sinyal genliğini yarıya indirir. Süre ve frekans konumları korunurken RMS yarıya düşer. RMS oranından hesaplanan seviye farkı:

$\Delta L=20\log_{10}(\mathrm{RMS}_{new}/\mathrm{RMS}_{original})\approx-6.02\ \mathrm{dB}$

Her spectrogramın kendi maksimumuna göre dB'ye çevrilmesi bu seviye farkını renk ölçeğinde gizleyebilir.

In [ ]:
if rms > 0:
    quieter = 0.5 * y
    quieter_rms = np.sqrt(np.mean(quieter.astype(np.float64)**2))
    delta_db = 20 * np.log10(quieter_rms / rms)
    print(f"Genlik ×0.5 → RMS oranı: {quieter_rms/rms:.3f}, değişim: {delta_db:.2f} dB")
else:
    print("Sessiz kayıtta RMS oranı tanımsızdır; bu deney atlandı.")


## Çıktı dosyaları

Grafikler `results/figures/01_audio_basics/` klasörüne SVG olarak kaydedilir. `summary.csv` ölçümleri, `run.json` giriş dosyasının SHA-256 değerini, analiz ayarlarını, zamanı ve paket sürümlerini içerir. Aynı çıktı yolları sonraki çalıştırmalarda güncellenir.

In [ ]:
metrics = {
    "sample_rate_hz": int(sr), "channels": int(audio.shape[1]),
    "samples_mono": len(y), "duration_s": duration,
    "peak_mono": peak, "rms_mono": rms,
    "window_ms": 1000*N_FFT/sr, "hop_ms": 1000*HOP_LENGTH/sr,
    "fft_bin_spacing_hz": sr/N_FFT,
    "stft_frequency_bins": int(power.shape[0]), "frames": int(power.shape[1]),
    "mel_bands": int(mel_power.shape[0]),
    "half_amplitude_rms_ratio": float(quieter_rms/rms) if rms > 0 else None,
    "half_amplitude_delta_db": float(delta_db) if rms > 0 else None,
}
with (TABLES / "summary.csv").open("w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["metric", "value"])
    writer.writerows(metrics.items())
try:
    source_path = str(AUDIO_PATH.resolve().relative_to(ROOT.resolve()))
except ValueError:
    source_path = AUDIO_PATH.name  # Dış dosyanın özel yerel yolunu kaydetme.
run = {
    "experiment_id": "01_audio_basics",
    "run_time_utc": datetime.now(timezone.utc).isoformat(),
    "source": source_path,
    "audio_sha256": hashlib.sha256(AUDIO_PATH.read_bytes()).hexdigest(),
    "python": platform.python_version(), "platform": platform.system(),
    "packages": {p: importlib.metadata.version(p) for p in
                 ["numpy", "scipy", "matplotlib", "librosa", "soundfile"]},
    "analysis": {"n_fft": N_FFT, "win_length": N_FFT,
                 "hop_length": HOP_LENGTH, "n_mels": N_MELS,
                 "window": "hann", "center": True, "fmin": 0,
                 "fmax": sr/2, "power_exponent": 2,
                 "db_reference": "per-representation maximum", "top_db": 80},
    "metrics": metrics,
    "scope": "audio representation demo; no trained model or paper reproduction",
}
(EXPERIMENT / "run.json").write_text(
    json.dumps(run, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
print("Kaydedildi:", FIGURES.relative_to(ROOT), TABLES.relative_to(ROOT),
      (EXPERIMENT / "run.json").relative_to(ROOT))
